# 00 - Setup: SoccerNet Game State Reconstruction (sn-gamestate)

This notebook sets up a Python 3.9 environment on Google Colab, installs [SoccerNet/sn-gamestate](https://github.com/SoccerNet/sn-gamestate) and its [TrackLab](https://github.com/TrackingLaboratory/tracklab) dependency, and runs the default game state reconstruction demo on a SoccerNet validation sequence.

**Run this in Google Colab** (not locally) — later cells assume a Colab VM.

## 1. Pin Python 3.9

sn-gamestate requires Python 3.9, but Colab's default runtime ships a newer Python. Colab has no built-in way to switch the notebook kernel to an arbitrary interpreter from the UI, so we install Python 3.9 via the deadsnakes PPA, create a venv, register it as a Jupyter kernel, and then manually switch this notebook to that kernel.

Run the cell below, then use Colab's kernel picker (the kernel/runtime name in the top-right of the notebook, *not* "Change runtime type") and select **Python 3.9 (sn-gamestate)**. Once switched, continue from the next cell.

In [ ]:
%%bash
set -e

# Add deadsnakes PPA and install Python 3.9
sudo apt-get update -qq
sudo apt-get install -y -qq software-properties-common
sudo add-apt-repository -y ppa:deadsnakes/ppa
sudo apt-get update -qq
sudo apt-get install -y -qq python3.9 python3.9-venv python3.9-dev

# Make sure the freshly installed interpreter has pip and a registered kernel,
# so the kernel doesn't hang on "Connecting" after a runtime restart.
python3.9 -m ensurepip --upgrade
python3.9 -m pip install ipykernel
python3.9 -m ipykernel install --name sn-gamestate-py39 --display-name "Python 3.9 (sn-gamestate)" --user

# Create a venv for the project and register it as a Jupyter kernel
python3.9 -m venv /content/env39
/content/env39/bin/pip install --quiet --upgrade pip ipykernel
/content/env39/bin/python -m ipykernel install --user --name sn-gamestate-py39 --display-name "Python 3.9 (sn-gamestate)"

echo "Python 3.9 kernel installed. Switch this notebook's kernel to 'Python 3.9 (sn-gamestate)' now."

In [ ]:
# Sanity check — run this AFTER switching the kernel to "Python 3.9 (sn-gamestate)"
import sys
print(sys.version)
assert sys.version_info[:2] == (3, 9), "Wrong kernel selected — pick 'Python 3.9 (sn-gamestate)' from the kernel picker."

## 2. Clone and install sn-gamestate + TrackLab

Following the official install instructions in the [sn-gamestate README](https://github.com/SoccerNet/sn-gamestate). We use `uv` to manage the environment; TrackLab is installed automatically as a dependency of `sn-gamestate`.

Colab presets `UV_SYSTEM_PYTHON=true` in the environment, which makes `uv` ignore any `.venv` it creates and install straight into the system Python (3.12) instead. That breaks this install, since `torch==1.13.1` has no Python 3.12 wheels — so the cell below unsets that variable and pins `uv` to the `.venv` interpreter explicitly.

In [ ]:
%%bash
set -e
unset UV_SYSTEM_PYTHON

# Install uv
curl -LsSf https://astral.sh/uv/install.sh | sh
export PATH="$HOME/.local/bin:$PATH"

cd /content
if [ ! -d sn-gamestate ]; then
  git clone https://github.com/SoccerNet/sn-gamestate.git
fi
cd sn-gamestate

uv venv --python 3.9
uv pip install -e . --python .venv/bin/python
uv run --python .venv/bin/python mim install mmcv==2.0.1

echo "sn-gamestate + TrackLab installed."

## 3. Dataset and model weights

This notebook does **not** bundle the SoccerNet dataset or model weights. TrackLab automatically downloads the SoccerNet-gamestate dataset and all required model weights the first time the pipeline is run below — this will take a while on first run (multiple GB) and requires accepting the SoccerNet dataset terms when prompted. Subsequent runs reuse the cached data under the `data_dir` configured in `sn_gamestate/configs/soccernet.yaml`.

## 4. Run the default game state reconstruction demo

Runs the default `soccernet` config, which processes one SoccerNet validation sequence end-to-end (detection, tracking, calibration, jersey/team classification, game state reconstruction).

In [ ]:
%%bash
set -e
export PATH="$HOME/.local/bin:$PATH"
cd /content/sn-gamestate
uv run tracklab -cn soccernet

## 5. Visualize the output

The demo writes an annotated video (tracking overlays + 2D pitch minimap) under `outputs/{date}/{time}/visualization/videos/`. The cell below finds the most recently produced video and displays it inline.

In [ ]:
import glob
import os
from IPython.display import Video

candidates = glob.glob("/content/sn-gamestate/outputs/**/visualization/videos/*.mp4", recursive=True)
assert candidates, "No output video found — check that the demo cell above completed successfully."
latest = max(candidates, key=os.path.getmtime)
print(f"Displaying: {latest}")
Video(latest, embed=True, width=800)